In [1]:
import pandas as pd 

# Load the dataset
data = pd.read_csv('data/movies.csv')

print(data.head())
print(data.shape)
print(data.columns)

                   title                genre  \
0           Harry Potter    fantasy adventure   
1             The Hobbit    fantasy adventure   
2  The Lord of the Rings    fantasy adventure   
3           Finding Nemo  animation adventure   
4              Toy Story     animation comedy   

                                         description  
0  A young wizard studies magic at a mysterious s...  
1  A hobbit joins a dangerous journey to find tre...  
2  Heroes travel across a fantasy world to destro...  
3      A fish searches the ocean for his missing son  
4  Toys come alive and discover friendship and ad...  
(30, 3)
Index(['title', 'genre', 'description'], dtype='str')


In [3]:
#explore the dataset
print("\nDataset information:")
print(data.info())
print("\nMissing values in each column:")
print(data.isnull().sum())
print("\nNumber of unique movies:")
print(data['title'].nunique())
print("\nMovie genres:")
print(data['genre'].head(10))
print("\nMovie description:")
print(data['description'].head(10))


Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   title        30 non-null     str  
 1   genre        30 non-null     str  
 2   description  30 non-null     str  
dtypes: str(3)
memory usage: 852.0 bytes
None

Missing values in each column:
title          0
genre          0
description    0
dtype: int64

Number of unique movies:
30

Movie genres:
0            fantasy adventure
1            fantasy adventure
2            fantasy adventure
3          animation adventure
4             animation comedy
5     animation comedy fantasy
6        science fiction drama
7    science fiction adventure
8     science fiction thriller
9       action science fiction
Name: genre, dtype: str

Movie description:
0    A young wizard studies magic at a mysterious s...
1    A hobbit joins a dangerous journey to find tre...
2    Heroes travel across a fantasy w

In [5]:
#feature engineering
data["content"]=(data["genre"]+" "+data["description"]).astype(str)
print("\nCombined movie content:")
print(data[['title', 'content']].head(10))



Combined movie content:
                   title                                            content
0           Harry Potter  fantasy adventure A young wizard studies magic...
1             The Hobbit  fantasy adventure A hobbit joins a dangerous j...
2  The Lord of the Rings  fantasy adventure Heroes travel across a fanta...
3           Finding Nemo  animation adventure A fish searches the ocean ...
4              Toy Story  animation comedy Toys come alive and discover ...
5                  Shrek  animation comedy fantasy An ogre goes on a fun...
6           Interstellar  science fiction drama Astronauts travel throug...
7            The Martian  science fiction adventure An astronaut struggl...
8                Gravity  science fiction thriller Astronauts fight to s...
9           The Avengers  action science fiction Superheroes unite to de...


In [5]:
#tf-idf vectorization
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
data = pd.read_csv("data/movies.csv")
data["content"]=(data["genre"]+" "+data["description"]).astype(str)
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(data['content'])
print("\nTF-IDF matrix shape:")
print(tfidf_matrix.shape)
print("\nVocabulary size:")
print(len(vectorizer.vocabulary_))
print("\nFirst movie vector:")
print(tfidf_matrix[0].toarray())


TF-IDF matrix shape:
(30, 144)

Vocabulary size:
144

First movie vector:
[[0.         0.         0.         0.16318958 0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.22306695 0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.40983306 0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.40983306 0.
  0.         0.         0

In [6]:
from sklearn.metrics.pairwise import cosine_similarity
# Compute the cosine similarity matrix
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
print("\nCosine similarity matrix shape:")
print(cosine_sim.shape)
print("\nHarry potter vs Lord of the Rings similarity:")
print(cosine_sim[0, 1])
print("\nHarry potter vs Harry potter similarity:")
print(cosine_sim[0, 0])


Cosine similarity matrix shape:
(30, 30)

Harry potter vs Lord of the Rings similarity:
0.08298667708740252

Harry potter vs Harry potter similarity:
0.9999999999999999


In [ ]:

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Step 1: Load the dataset
data = pd.read_csv("data/movies.csv")

# Step 2: Prepare movie content
data["content"] = (
    data["genre"].fillna("").astype(str) + " " +
    data["description"].fillna("").astype(str)
)

# Step 3: Convert movie text into TF-IDF vectors
vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = vectorizer.fit_transform(data["content"])

# Step 4: Calculate similarity between all movies
similarity_matrix = cosine_similarity(tfidf_matrix)

print("Similarity matrix shape:", similarity_matrix.shape)
print("Total movies:", len(data))


# Step 5: Create the recommendation function
def recommend_movies(movie_title, top_n=5):

    # Find the movie entered by the user
    matches = data[
        data["title"].str.strip().str.lower()
        == movie_title.strip().lower()
    ]

    if matches.empty:
        print("Movie not found. Please check the title.")
        return

    # Get the movie's position in the dataset
    movie_index = matches.index[0]

    # Get similarity scores for this movie
    scores = list(enumerate(similarity_matrix[movie_index]))

    # Sort from most similar to least similar
    scores = sorted(
        scores,
        key=lambda item: item[1],
        reverse=True
    )

    # Remove the selected movie itself
    scores = [
        item for item in scores
        if item[0] != movie_index
    ]

    # Display top recommendations
    top_scores = scores[:top_n]

    print(f"\nMovies similar to '{movie_title}':")

    for index, score in top_scores:
        print(
            f"{data.iloc[index]['title']} "
            f"(similarity: {score:.3f})"
        )


# Step 6: Test the recommendation system
recommend_movies("The shrek")


Similarity matrix shape: (30, 30)
Total movies: 30

Movies similar to 'The Hobbit':
Pirates of the Caribbean (similarity: 0.224)
Batman Begins (similarity: 0.135)
The Lord of the Rings (similarity: 0.134)
Shrek (similarity: 0.086)
How to Train Your Dragon (similarity: 0.083)
